# Numerai v5.2 — Selection + PCA500 + Lag1/Lag2 + Ridge/ElasticNet/XGBoost Stacking

이 노트북은 기존 Ridge + ElasticNet 전체 파이프라인의 구조와 변수명을 최대한 유지하면서 XGBoost를 추가합니다.

파이프라인:

1. `v5.2/features.json`, `train.parquet`, `validation.parquet` 다운로드
2. 전체 feature 중 target 절대상관계수 상위 50% 선택
3. `StandardScaler + IncrementalPCA(500)` 학습 및 저장
4. `PCA500 + era 평균 lag1 + lag2` 생성
5. 내부 데이터에서 Ridge + ElasticNet + XGBoost 학습
6. 세 모델의 era별 rank를 입력으로 `Ridge(alpha=1.0)` 메타모델 학습
7. OOS 검증 후 기본 모델을 전체 내부 데이터로 재학습
8. 공식 `validation.parquet`을 동일하게 변환
9. Numerai Diagnostics용 `id,prediction` CSV 생성

중요:
- 기존 전처리 결과와 맞추기 위해 `VERSION = "v5.2"`, `TARGET_COL = "target"`로 고정합니다.
- Ridge와 ElasticNet에는 모델 입력용 StandardScaler를 적용합니다.
- XGBoost는 같은 1,500개 feature를 사용하지만 트리 모델이므로 별도 scaling 없이 학습합니다.
- XGBoost 하이퍼파라미터는 기존 Optuna 1차 탐색 최적값을 사용합니다.

In [2]:
# 필요한 패키지 설치 — Jupyter/Colab에서 한 번만 실행
# Colab의 pandas 고정 버전 충돌을 피하기 위해 pandas는 강제 업그레이드하지 않습니다.
%pip install -q numerapi pyarrow scikit-learn scipy joblib cloudpickle numerai-tools xgboost

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================
# 0. 라이브러리 / 경로 / 공통 설정
# ============================================
import gc
import json
import os
import random
import shutil
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import xgboost as xgb

from numerapi import NumerAPI
from numerai_tools.scoring import numerai_corr
from sklearn.decomposition import IncrementalPCA
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

VERSION = "v5.2"
TARGET_COL = "target"
PCA_N = 500
LAGS = (1, 2)

SELECTION_BATCH_SIZE = 20_000
PCA_FIT_BATCH_SIZE = 50_000
TRANSFORM_BATCH_SIZE = 20_000
MODEL_BATCH_SIZE = 50_000

# 기존 Mac 로컬 경로를 그대로 사용합니다.
# Colab Drive를 쓸 경우 이 한 줄만 Drive 경로로 바꾸면 됩니다.
BASE_DIR = Path(
    "/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai"
)

DATA_DIR = BASE_DIR / VERSION
PREPROCESS_DIR = BASE_DIR / "numerai_postprocess_v52"
CACHE_DIR = BASE_DIR / "cache_v52"
MODEL_DIR = BASE_DIR / "ridge_elastic_xgboost_stack_v52"
DIAGNOSTICS_DIR = BASE_DIR / "diagnostics_v52"
MEMMAP_DIR = MODEL_DIR / "memmap"

for directory in [
    DATA_DIR,
    PREPROCESS_DIR,
    CACHE_DIR,
    MODEL_DIR,
    DIAGNOSTICS_DIR,
    MEMMAP_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

FEATURES_JSON_PATH = DATA_DIR / "features.json"
TRAIN_PATH = DATA_DIR / "train.parquet"
VALIDATION_PATH = DATA_DIR / "validation.parquet"

TRAIN_PCA_ALL_PATH = CACHE_DIR / "train_pca_all.parquet"
TRAIN_FINAL_PATH = CACHE_DIR / "train_final_cache.parquet"
VAL_FINAL_PATH = CACHE_DIR / "val_final_cache.parquet"
LEGACY_MODEL_DATA_PATH = BASE_DIR / "pca500_lag1_lag2.parquet"

VALIDATION_PCA_ALL_PATH = CACHE_DIR / "validation_pca_all.parquet"
VALIDATION_FINAL_PATH = CACHE_DIR / "validation_pca500_lag1_lag2.parquet"


def detect_xgb_device():
    # NVIDIA GPU가 실제로 보이면 cuda, 아니면 cpu를 사용합니다.
    try:
        result = subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        )
        return "cuda" if result.returncode == 0 else "cpu"
    except FileNotFoundError:
        return "cpu"


XGB_DEVICE = detect_xgb_device()

print("BASE_DIR:", BASE_DIR)
print("VERSION:", VERSION)
print("XGBoost device:", XGB_DEVICE)

BASE_DIR: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai
VERSION: v5.2
XGBoost device: cpu


In [4]:
# ============================================
# 1. 데이터 다운로드
# ============================================
napi = NumerAPI()


def download_with_retry(filename, max_retries=5):
    target = BASE_DIR / filename
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists():
        print("이미 존재:", target)
        return target

    for attempt in range(1, max_retries + 1):
        try:
            print(f"다운로드 {attempt}/{max_retries}: {filename}")
            napi.download_dataset(filename, dest_path=str(target))
            return target
        except Exception as error:
            print("다운로드 실패:", error)
            if attempt == max_retries:
                raise


for filename in [
    f"{VERSION}/features.json",
    f"{VERSION}/train.parquet",
    f"{VERSION}/validation.parquet",
]:
    download_with_retry(filename)

print("features.json:", FEATURES_JSON_PATH.exists())
print("train.parquet:", TRAIN_PATH.exists())
print("validation.parquet:", VALIDATION_PATH.exists())

이미 존재: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/v5.2/features.json
이미 존재: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/v5.2/train.parquet
이미 존재: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/v5.2/validation.parquet
features.json: True
train.parquet: True
validation.parquet: True


In [5]:
# ============================================
# 2. 공용 스트리밍 유틸
# ============================================
def clean_array(arr):
    return np.nan_to_num(
        arr,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


def get_schema_names(path):
    return pq.ParquetFile(path).schema_arrow.names


def resolve_id_storage_column(path):
    names = get_schema_names(path)

    if "id" in names:
        return "id"
    if "__index_level_0__" in names:
        return "__index_level_0__"

    return None


def normalize_id_column(chunk):
    if "id" in chunk.columns:
        return chunk
    if "__index_level_0__" in chunk.columns:
        return chunk.rename(columns={"__index_level_0__": "id"})
    if chunk.index.name == "id":
        return chunk.reset_index()

    raise ValueError(
        "id를 찾지 못했습니다. "
        f"columns={chunk.columns.tolist()[:10]}, index={chunk.index.name}"
    )


def iter_parquet_batches(path, columns, batch_size):
    pf = pq.ParquetFile(path)
    available = set(pf.schema_arrow.names)

    requested = []
    for column in columns:
        if column in available and column not in requested:
            requested.append(column)

    id_storage = resolve_id_storage_column(path)
    if "id" in columns and id_storage is not None and id_storage not in requested:
        requested.insert(0, id_storage)

    for batch in pf.iter_batches(
        columns=requested,
        batch_size=batch_size,
    ):
        chunk = batch.to_pandas()

        if "id" in columns:
            chunk = normalize_id_column(chunk)

        if "era" in chunk.columns:
            chunk["era"] = chunk["era"].astype(int)

        yield chunk


def remove_if_exists(path):
    path = Path(path)
    if path.exists():
        path.unlink()


def append_parquet_batch(writer, frame):
    # id를 pandas index로 저장하여 기존 캐시 구조를 유지합니다.
    frame = frame.set_index("id")
    table = pa.Table.from_pandas(frame, preserve_index=True)

    if writer["object"] is None:
        writer["object"] = pq.ParquetWriter(
            str(writer["path"]),
            table.schema,
            compression="snappy",
        )

    writer["object"].write_table(table)


def close_writer(writer):
    if writer["object"] is not None:
        writer["object"].close()
        writer["object"] = None

In [6]:
# ============================================
# 3. Feature Selection
# 전체 all feature 중 |corr(feature, target)| 상위 50%
# ============================================
with open(FEATURES_JSON_PATH, "r", encoding="utf-8") as file:
    feature_metadata = json.load(file)

all_feature_cols = feature_metadata["feature_sets"]["all"]
print("전체 feature 수:", len(all_feature_cols))

SELECTED_FEATURES_PATH = PREPROCESS_DIR / "selected_features.json"

if SELECTED_FEATURES_PATH.exists():
    with open(SELECTED_FEATURES_PATH, "r", encoding="utf-8") as file:
        selected_features = json.load(file)

    print("기존 selected_features 로드:", len(selected_features))

else:
    p = len(all_feature_cols)

    sum_x = np.zeros(p, dtype=np.float64)
    sum_x2 = np.zeros(p, dtype=np.float64)
    sum_xy = np.zeros(p, dtype=np.float64)
    sum_y = 0.0
    sum_y2 = 0.0
    n_total = 0

    selection_columns = all_feature_cols + [TARGET_COL]

    for chunk in iter_parquet_batches(
        TRAIN_PATH,
        selection_columns,
        SELECTION_BATCH_SIZE,
    ):
        chunk = chunk[chunk[TARGET_COL].notna()].copy()
        if chunk.empty:
            continue

        X_chunk = clean_array(
            chunk[all_feature_cols].to_numpy(dtype=np.float32)
        )
        y_chunk = clean_array(
            chunk[TARGET_COL].to_numpy(dtype=np.float32)
        )

        n_chunk = len(y_chunk)
        n_total += n_chunk

        sum_x += X_chunk.sum(axis=0, dtype=np.float64)
        sum_x2 += (X_chunk * X_chunk).sum(axis=0, dtype=np.float64)
        sum_xy += X_chunk.T @ y_chunk.astype(np.float64)
        sum_y += y_chunk.sum(dtype=np.float64)
        sum_y2 += (y_chunk * y_chunk).sum(dtype=np.float64)

        print(f"Feature selection: {n_total:,} rows")

    cov_xy = sum_xy - (sum_x * sum_y / n_total)
    var_x = sum_x2 - (sum_x ** 2 / n_total)
    var_y = sum_y2 - (sum_y ** 2 / n_total)

    corr = cov_xy / (np.sqrt(var_x * var_y) + 1e-12)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)

    scores = np.abs(corr)
    threshold = np.median(scores)

    selected_features = [
        feature
        for feature, score in zip(all_feature_cols, scores)
        if score >= threshold
    ]

    with open(SELECTED_FEATURES_PATH, "w", encoding="utf-8") as file:
        json.dump(selected_features, file, ensure_ascii=False, indent=2)

    print("selected_features 저장:", SELECTED_FEATURES_PATH)

print("선택된 feature 수:", len(selected_features))

if VERSION == "v5.2" and len(selected_features) != 1374:
    print("주의: 기존 v5.2 결과는 1,374개였습니다.")

전체 feature 수: 2748
기존 selected_features 로드: 1374
선택된 feature 수: 1374


In [7]:
# ============================================
# 4. Raw StandardScaler + IncrementalPCA(500)
# ============================================
RAW_SCALER_PATH = PREPROCESS_DIR / "fitted_scaler.joblib"
PCA_PATH = PREPROCESS_DIR / "fitted_pca.joblib"

if RAW_SCALER_PATH.exists() and PCA_PATH.exists():
    raw_scaler = joblib.load(RAW_SCALER_PATH)
    pca = joblib.load(PCA_PATH)
    print("기존 scaler / PCA 로드 완료")

else:
    fit_columns = selected_features

    raw_scaler = StandardScaler()
    processed = 0

    for chunk in iter_parquet_batches(
        TRAIN_PATH,
        fit_columns,
        PCA_FIT_BATCH_SIZE,
    ):
        X_chunk = clean_array(
            chunk[selected_features].to_numpy(dtype=np.float32)
        )
        raw_scaler.partial_fit(X_chunk)
        processed += len(chunk)
        print(f"[Raw Scaler] {processed:,} rows")

    pca_n = min(PCA_N, len(selected_features))
    pca = IncrementalPCA(
        n_components=pca_n,
        batch_size=PCA_FIT_BATCH_SIZE,
    )

    processed = 0
    fitted_batches = 0

    for chunk in iter_parquet_batches(
        TRAIN_PATH,
        fit_columns,
        PCA_FIT_BATCH_SIZE,
    ):
        X_chunk = clean_array(
            chunk[selected_features].to_numpy(dtype=np.float32)
        )
        X_scaled = raw_scaler.transform(X_chunk)

        if X_scaled.shape[0] >= pca_n:
            pca.partial_fit(X_scaled)
            fitted_batches += 1

        processed += len(chunk)
        print(f"[PCA] {processed:,} rows")

    if fitted_batches == 0:
        raise RuntimeError("PCA가 한 번도 fit되지 않았습니다.")

    joblib.dump(raw_scaler, RAW_SCALER_PATH)
    joblib.dump(pca, PCA_PATH)

    print("raw scaler 저장:", RAW_SCALER_PATH)
    print("PCA 저장:", PCA_PATH)

print("selected feature 수:", len(selected_features))
print("raw scaler 입력 수:", raw_scaler.n_features_in_)
print("PCA 입력 수:", pca.n_features_in_)
print("PCA component 수:", pca.n_components_)
print("설명분산 비율 합:", pca.explained_variance_ratio_.sum())

assert len(selected_features) == raw_scaler.n_features_in_
assert len(selected_features) == pca.n_features_in_
assert pca.n_components_ == PCA_N

기존 scaler / PCA 로드 완료
selected feature 수: 1374
raw scaler 입력 수: 1374
PCA 입력 수: 1374
PCA component 수: 500
설명분산 비율 합: 0.9041530365450252


In [8]:
# ============================================
# 5. 원본 데이터 → PCA500 변환
# ============================================
pca_cols = [f"pca_{i}" for i in range(PCA_N)]


def transform_source_to_pca(
    source_path,
    output_path,
    filter_validation=False,
):
    remove_if_exists(output_path)
    writer = {"path": Path(output_path), "object": None}

    columns = ["id", "era", TARGET_COL] + selected_features
    if filter_validation:
        columns.insert(2, "data_type")

    processed = 0

    try:
        for chunk in iter_parquet_batches(
            source_path,
            columns,
            TRANSFORM_BATCH_SIZE,
        ):
            if filter_validation:
                chunk = chunk[chunk["data_type"] == "validation"].copy()
                if chunk.empty:
                    continue

            X_raw = clean_array(
                chunk[selected_features].to_numpy(dtype=np.float32)
            )
            X_scaled = raw_scaler.transform(X_raw)
            X_pca = pca.transform(X_scaled).astype(np.float32)

            out = pd.DataFrame(X_pca, columns=pca_cols)
            out["era"] = chunk["era"].to_numpy(dtype=np.int32)
            out[TARGET_COL] = chunk[TARGET_COL].to_numpy(dtype=np.float32)
            out["id"] = chunk["id"].astype(str).to_numpy()
            out = out[["id"] + pca_cols + ["era", TARGET_COL]]

            append_parquet_batch(writer, out)
            processed += len(out)
            print(f"PCA 변환: {processed:,} rows")

    finally:
        close_writer(writer)

    print("저장 완료:", output_path)


if not TRAIN_PCA_ALL_PATH.exists():
    transform_source_to_pca(
        TRAIN_PATH,
        TRAIN_PCA_ALL_PATH,
        filter_validation=False,
    )
else:
    print("기존 파일 사용:", TRAIN_PCA_ALL_PATH)

기존 파일 사용: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/cache_v52/train_pca_all.parquet


In [9]:
# ============================================
# 6. Era split + era 평균 lag1/lag2
# train_final_cache / val_final_cache 생성
# ============================================
def compute_era_means(pca_path):
    sums = {}
    counts = {}

    for chunk in iter_parquet_batches(
        pca_path,
        ["era"] + pca_cols,
        TRANSFORM_BATCH_SIZE,
    ):
        era_array = chunk["era"].to_numpy(dtype=np.int32)
        X = chunk[pca_cols].to_numpy(dtype=np.float32)

        for era in np.unique(era_array):
            mask = era_array == era
            current_sum = X[mask].sum(axis=0, dtype=np.float64)
            current_count = int(mask.sum())

            if int(era) not in sums:
                sums[int(era)] = current_sum
                counts[int(era)] = current_count
            else:
                sums[int(era)] += current_sum
                counts[int(era)] += current_count

    eras = sorted(sums)
    means = np.vstack([
        sums[era] / counts[era]
        for era in eras
    ]).astype(np.float32)

    return pd.DataFrame(means, index=eras, columns=pca_cols)


def make_lag_lookup(era_mean_df, era_subset):
    subset = era_mean_df.loc[sorted(era_subset)].copy()

    lag1 = subset.shift(1).fillna(0)
    lag2 = subset.shift(2).fillna(0)

    lag1_lookup = {
        int(era): row.to_numpy(dtype=np.float32)
        for era, row in lag1.iterrows()
    }
    lag2_lookup = {
        int(era): row.to_numpy(dtype=np.float32)
        for era, row in lag2.iterrows()
    }

    return lag1_lookup, lag2_lookup


era_mean_df = compute_era_means(TRAIN_PCA_ALL_PATH)
unique_eras = sorted(era_mean_df.index.astype(int).tolist())

split_point = int(len(unique_eras) * 0.8)
train_eras = set(unique_eras[:split_point])
valid_eras = set(unique_eras[split_point:])

print("전체 era:", len(unique_eras))
print("train era:", len(train_eras), min(train_eras), "~", max(train_eras))
print("valid era:", len(valid_eras), min(valid_eras), "~", max(valid_eras))

train_lag1, train_lag2 = make_lag_lookup(era_mean_df, train_eras)
valid_lag1, valid_lag2 = make_lag_lookup(era_mean_df, valid_eras)

lag1_cols = [f"{column}_lag1" for column in pca_cols]
lag2_cols = [f"{column}_lag2" for column in pca_cols]
FEATURES = pca_cols + lag1_cols + lag2_cols

remove_if_exists(TRAIN_FINAL_PATH)
remove_if_exists(VAL_FINAL_PATH)

train_writer = {"path": TRAIN_FINAL_PATH, "object": None}
valid_writer = {"path": VAL_FINAL_PATH, "object": None}

train_rows = 0
valid_rows = 0

try:
    for chunk in iter_parquet_batches(
        TRAIN_PCA_ALL_PATH,
        ["id", "era", TARGET_COL] + pca_cols,
        TRANSFORM_BATCH_SIZE,
    ):
        for era_set, lag1_lookup, lag2_lookup, writer_name in [
            (train_eras, train_lag1, train_lag2, "train"),
            (valid_eras, valid_lag1, valid_lag2, "valid"),
        ]:
            sub = chunk[chunk["era"].isin(era_set)].copy()
            if sub.empty:
                continue

            era_array = sub["era"].to_numpy(dtype=np.int32)
            lag1_array = np.vstack([
                lag1_lookup[int(era)]
                for era in era_array
            ])
            lag2_array = np.vstack([
                lag2_lookup[int(era)]
                for era in era_array
            ])

            final = pd.concat(
                [
                    sub[["id"] + pca_cols].reset_index(drop=True),
                    pd.DataFrame(lag1_array, columns=lag1_cols),
                    pd.DataFrame(lag2_array, columns=lag2_cols),
                ],
                axis=1,
            )

            final["era"] = era_array
            final[TARGET_COL] = sub[TARGET_COL].to_numpy(dtype=np.float32)
            final = final[["id"] + FEATURES + ["era", TARGET_COL]]

            if writer_name == "train":
                append_parquet_batch(train_writer, final)
                train_rows += len(final)
            else:
                append_parquet_batch(valid_writer, final)
                valid_rows += len(final)

        print(f"cache 생성: train={train_rows:,}, valid={valid_rows:,}")

finally:
    close_writer(train_writer)
    close_writer(valid_writer)

# 기존 모델 노트북이 사용한 이름을 그대로 생성합니다.
shutil.copy2(VAL_FINAL_PATH, LEGACY_MODEL_DATA_PATH)

print("train_final_cache:", TRAIN_FINAL_PATH)
print("val_final_cache:", VAL_FINAL_PATH)
print("pca500_lag1_lag2:", LEGACY_MODEL_DATA_PATH)

전체 era: 574
train era: 459 1 ~ 459
valid era: 115 460 ~ 574
cache 생성: train=20,000, valid=0
cache 생성: train=40,000, valid=0
cache 생성: train=60,000, valid=0
cache 생성: train=80,000, valid=0
cache 생성: train=100,000, valid=0
cache 생성: train=120,000, valid=0
cache 생성: train=140,000, valid=0
cache 생성: train=160,000, valid=0
cache 생성: train=180,000, valid=0
cache 생성: train=200,000, valid=0
cache 생성: train=220,000, valid=0
cache 생성: train=240,000, valid=0
cache 생성: train=260,000, valid=0
cache 생성: train=280,000, valid=0
cache 생성: train=300,000, valid=0
cache 생성: train=320,000, valid=0
cache 생성: train=340,000, valid=0
cache 생성: train=360,000, valid=0
cache 생성: train=380,000, valid=0
cache 생성: train=400,000, valid=0
cache 생성: train=420,000, valid=0
cache 생성: train=440,000, valid=0
cache 생성: train=460,000, valid=0
cache 생성: train=480,000, valid=0
cache 생성: train=500,000, valid=0
cache 생성: train=520,000, valid=0
cache 생성: train=540,000, valid=0
cache 생성: train=560,000, valid=0
cache 생성: train=580,

In [10]:
# ============================================
# 7. 캐시 구조 확인
# ============================================
train_final_check = pd.read_parquet(
    TRAIN_FINAL_PATH,
    columns=["era", TARGET_COL],
)
val_final_check = pd.read_parquet(
    VAL_FINAL_PATH,
    columns=["era", TARGET_COL],
)

print(
    "train_final shape:",
    pq.ParquetFile(TRAIN_FINAL_PATH).metadata.num_rows,
    "rows",
)
print(
    "val_final shape:",
    pq.ParquetFile(VAL_FINAL_PATH).metadata.num_rows,
    "rows",
)
print("train era:", train_final_check["era"].min(), "~", train_final_check["era"].max())
print("valid era:", val_final_check["era"].min(), "~", val_final_check["era"].max())

sample = pd.read_parquet(VAL_FINAL_PATH).head()
print("id index name:", sample.index.name)
print("feature 수:", len(FEATURES))
print("뒤 컬럼:", sample.columns[-10:].tolist())

assert sample.index.name == "id"
assert len(FEATURES) == 1500

# 확인용 객체는 메모리에서 제거합니다.
del train_final_check, val_final_check, sample
gc.collect()

train_final shape: 2158685 rows
val_final shape: 587583 rows
train era: 1 ~ 459
valid era: 460 ~ 574
id index name: id
feature 수: 1500
뒤 컬럼: ['pca_492_lag2', 'pca_493_lag2', 'pca_494_lag2', 'pca_495_lag2', 'pca_496_lag2', 'pca_497_lag2', 'pca_498_lag2', 'pca_499_lag2', 'era', 'target']


0

In [11]:
# ============================================
# 8. Ridge / ElasticNet / XGBoost 내부 학습 데이터 준비
# 기존 코드와 동일하게 587,583행 데이터 사용
# ============================================
model_df = pd.read_parquet(LEGACY_MODEL_DATA_PATH)

model_df[FEATURES] = (
    model_df[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype(np.float32)
)
model_df[TARGET_COL] = model_df[TARGET_COL].astype(np.float32)
model_df = model_df[model_df[TARGET_COL].notna()].copy()

internal_eras = sorted(model_df["era"].unique())
internal_split = int(len(internal_eras) * 0.8)

internal_train_eras = internal_eras[:internal_split]
internal_valid_eras = internal_eras[internal_split:]

train_df = (
    model_df[model_df["era"].isin(internal_train_eras)]
    .reset_index()
)
valid_df = (
    model_df[model_df["era"].isin(internal_valid_eras)]
    .reset_index()
)

X_train = train_df[FEATURES].to_numpy(dtype=np.float32)
y_train_arr = train_df[TARGET_COL].to_numpy(dtype=np.float32)

X_valid = valid_df[FEATURES].to_numpy(dtype=np.float32)
y_valid_arr = valid_df[TARGET_COL].to_numpy(dtype=np.float32)

val_era_arr = valid_df["era"].to_numpy(dtype=np.int32)
val_id_arr = valid_df["id"].astype(str).to_numpy()

print("전체 era:", len(internal_eras))
print("train era:", min(internal_train_eras), "~", max(internal_train_eras))
print("valid era:", min(internal_valid_eras), "~", max(internal_valid_eras))
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

# 필요한 배열만 남겨 메모리를 확보합니다.
del train_df, valid_df, model_df
gc.collect()

전체 era: 115
train era: 460 ~ 551
valid era: 552 ~ 574
X_train: (461940, 1500)
X_valid: (125643, 1500)


0

In [12]:
# ============================================
# 9. 모델 입력용 Scaling
# Ridge / ElasticNet만 scaling하고 XGBoost는 원본 1,500개 feature 사용
# ============================================
model_scaler = StandardScaler()

for start in range(0, X_train.shape[0], MODEL_BATCH_SIZE):
    end = min(start + MODEL_BATCH_SIZE, X_train.shape[0])
    model_scaler.partial_fit(X_train[start:end])
    print(f"Model scaler fit: {end:,}/{X_train.shape[0]:,}")

X_TRAIN_SCALED_PATH = MEMMAP_DIR / "X_train_scaled.dat"
X_VALID_SCALED_PATH = MEMMAP_DIR / "X_valid_scaled.dat"

X_train_scaled = np.memmap(
    X_TRAIN_SCALED_PATH,
    dtype="float32",
    mode="w+",
    shape=X_train.shape,
)
X_valid_scaled = np.memmap(
    X_VALID_SCALED_PATH,
    dtype="float32",
    mode="w+",
    shape=X_valid.shape,
)

for start in range(0, X_train.shape[0], MODEL_BATCH_SIZE):
    end = min(start + MODEL_BATCH_SIZE, X_train.shape[0])
    X_train_scaled[start:end] = model_scaler.transform(
        X_train[start:end]
    ).astype(np.float32)
    print(f"X_train scaling: {end:,}/{X_train.shape[0]:,}")

for start in range(0, X_valid.shape[0], MODEL_BATCH_SIZE):
    end = min(start + MODEL_BATCH_SIZE, X_valid.shape[0])
    X_valid_scaled[start:end] = model_scaler.transform(
        X_valid[start:end]
    ).astype(np.float32)
    print(f"X_valid scaling: {end:,}/{X_valid.shape[0]:,}")

X_train_scaled.flush()
X_valid_scaled.flush()

joblib.dump(model_scaler, MODEL_DIR / "model_input_scaler.joblib")
print("model scaler 저장 완료")

Model scaler fit: 50,000/461,940
Model scaler fit: 100,000/461,940
Model scaler fit: 150,000/461,940
Model scaler fit: 200,000/461,940
Model scaler fit: 250,000/461,940
Model scaler fit: 300,000/461,940
Model scaler fit: 350,000/461,940
Model scaler fit: 400,000/461,940
Model scaler fit: 450,000/461,940
Model scaler fit: 461,940/461,940
X_train scaling: 50,000/461,940
X_train scaling: 100,000/461,940
X_train scaling: 150,000/461,940
X_train scaling: 200,000/461,940
X_train scaling: 250,000/461,940
X_train scaling: 300,000/461,940
X_train scaling: 350,000/461,940
X_train scaling: 400,000/461,940
X_train scaling: 450,000/461,940
X_train scaling: 461,940/461,940
X_valid scaling: 50,000/125,643
X_valid scaling: 100,000/125,643
X_valid scaling: 125,643/125,643
model scaler 저장 완료


In [13]:
# ============================================
# 10. 평가 함수
# ============================================
def simple_per_era_corr(era_arr, target_arr, pred_arr):
    eval_df = pd.DataFrame({
        "era": era_arr,
        "target": target_arr,
        "prediction": pred_arr,
    })

    def era_corr(sub):
        return np.corrcoef(
            sub["prediction"].rank(pct=True),
            sub["target"],
        )[0, 1]

    try:
        return eval_df.groupby("era").apply(
            era_corr,
            include_groups=False,
        )
    except TypeError:
        # pandas 구버전 호환
        return eval_df.groupby("era").apply(era_corr)


def evaluate_prediction(name, prediction):
    prediction = np.asarray(prediction).reshape(-1)

    corr = simple_per_era_corr(
        val_era_arr,
        y_valid_arr,
        prediction,
    )

    mean_corr = corr.mean()
    std_corr = corr.std(ddof=0)
    sharpe = mean_corr / std_corr if std_corr > 0 else np.nan

    rmse = np.sqrt(mean_squared_error(y_valid_arr, prediction))
    mae = mean_absolute_error(y_valid_arr, prediction)

    print()
    print(f"========== {name} Validation Result ==========")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE: {mae:.6f}")
    print(f"per-era CORR mean: {mean_corr:.6f}")
    print(f"per-era CORR std:  {std_corr:.6f}")
    print(f"Sharpe:            {sharpe:.6f}")
    print("================================================")

    return {
        "model": name,
        "rmse": rmse,
        "mae": mae,
        "mean_corr": mean_corr,
        "std_corr": std_corr,
        "sharpe": sharpe,
    }

In [14]:
# ============================================
# 11. Base Ridge + ElasticNet + XGBoost 학습
# ============================================
# Ridge — 기존 탐색 결과에서 가장 높은 Sharpe를 보인 alpha
BASE_RIDGE_ALPHA = 3_000_000

base_ridge_model = Ridge(
    alpha=BASE_RIDGE_ALPHA,
    fit_intercept=True,
    solver="lsqr",
    max_iter=5000,
    tol=1e-4,
)
base_ridge_model.fit(X_train_scaled, y_train_arr)
ridge_pred = base_ridge_model.predict(X_valid_scaled).astype(np.float32)


# ElasticNet — 기존 코드와 동일한 SGDRegressor 방식
base_elastic_model = SGDRegressor(
    loss="squared_error",
    penalty="elasticnet",
    alpha=1e-3,
    l1_ratio=0.03,
    learning_rate="invscaling",
    eta0=0.001,
    power_t=0.5,
    max_iter=1,
    tol=None,
    random_state=SEED,
    average=True,
)

ELASTIC_EPOCHS = 20
rng = np.random.default_rng(SEED)

for epoch in range(ELASTIC_EPOCHS):
    indices = rng.permutation(X_train_scaled.shape[0])

    for start in range(0, len(indices), MODEL_BATCH_SIZE):
        end = min(start + MODEL_BATCH_SIZE, len(indices))
        idx = indices[start:end]

        base_elastic_model.partial_fit(
            np.asarray(X_train_scaled[idx], dtype=np.float32),
            y_train_arr[idx],
        )

    print(f"ElasticNet epoch {epoch + 1}/{ELASTIC_EPOCHS} 완료")

elastic_pred = base_elastic_model.predict(X_valid_scaled).astype(np.float32)


# XGBoost — 기존 Optuna 1차 탐색 최적값
XGB_BEST_PARAMS = {
    "n_estimators": 2000,
    "learning_rate": 0.016837,
    "max_depth": 3,
    "subsample": 0.684226,
    "colsample_bytree": 0.916064,
    "reg_alpha": 0.271880,
    "reg_lambda": 6.848673,
    "min_child_weight": 7,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "random_state": SEED,
    "tree_method": "hist",
    "device": XGB_DEVICE,
    "n_jobs": -1,
    "early_stopping_rounds": 100,
}


def fit_xgb_with_fallback(params, X_fit, y_fit, X_eval, y_eval):
    model = xgb.XGBRegressor(**params)

    try:
        model.fit(
            X_fit,
            y_fit,
            eval_set=[(X_eval, y_eval)],
            verbose=100,
        )
        return model

    except Exception as error:
        if params.get("device") != "cuda":
            raise

        print("CUDA 학습 실패 — CPU로 다시 실행합니다.")
        print("원래 오류:", error)

        cpu_params = params.copy()
        cpu_params["device"] = "cpu"

        model = xgb.XGBRegressor(**cpu_params)
        model.fit(
            X_fit,
            y_fit,
            eval_set=[(X_eval, y_eval)],
            verbose=100,
        )
        return model


base_xgb_model = fit_xgb_with_fallback(
    XGB_BEST_PARAMS,
    X_train,
    y_train_arr,
    X_valid,
    y_valid_arr,
)

xgb_pred = base_xgb_model.predict(X_valid).astype(np.float32)

ridge_result = evaluate_prediction("ridge", ridge_pred)
elastic_result = evaluate_prediction("elasticnet", elastic_pred)
xgb_result = evaluate_prediction("xgboost", xgb_pred)

base_results = pd.DataFrame([
    ridge_result,
    elastic_result,
    xgb_result,
]).sort_values("sharpe", ascending=False).reset_index(drop=True)

display(base_results)

joblib.dump(base_ridge_model, MODEL_DIR / "base_ridge_model.joblib")
joblib.dump(base_elastic_model, MODEL_DIR / "base_elasticnet_model.joblib")
joblib.dump(base_xgb_model, MODEL_DIR / "base_xgboost_model.joblib")
base_xgb_model.save_model(MODEL_DIR / "base_xgboost_model.json")

# 기존 CSV 형태 유지
pd.DataFrame({
    "era": val_era_arr,
    "prediction": ridge_pred,
}).to_csv(
    MODEL_DIR / "ridge_val_prediction.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame({
    "era": val_era_arr,
    "prediction": elastic_pred,
}).to_csv(
    MODEL_DIR / "elasticnet_val_prediction.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame({
    "era": val_era_arr,
    "prediction": xgb_pred,
}).to_csv(
    MODEL_DIR / "xgboost_val_prediction.csv",
    index=False,
    encoding="utf-8-sig",
)

# id와 target이 포함된 안전한 버전
for name, prediction in [
    ("ridge", ridge_pred),
    ("elasticnet", elastic_pred),
    ("xgboost", xgb_pred),
]:
    pd.DataFrame({
        "id": val_id_arr,
        "era": val_era_arr,
        "target": y_valid_arr,
        "prediction": prediction,
    }).to_csv(
        MODEL_DIR / f"{name}_realtarget_val_predictions.csv",
        index=False,
    )

ElasticNet epoch 1/20 완료
ElasticNet epoch 2/20 완료
ElasticNet epoch 3/20 완료
ElasticNet epoch 4/20 완료
ElasticNet epoch 5/20 완료
ElasticNet epoch 6/20 완료
ElasticNet epoch 7/20 완료
ElasticNet epoch 8/20 완료
ElasticNet epoch 9/20 완료
ElasticNet epoch 10/20 완료
ElasticNet epoch 11/20 완료
ElasticNet epoch 12/20 완료
ElasticNet epoch 13/20 완료
ElasticNet epoch 14/20 완료
ElasticNet epoch 15/20 완료
ElasticNet epoch 16/20 완료
ElasticNet epoch 17/20 완료
ElasticNet epoch 18/20 완료
ElasticNet epoch 19/20 완료
ElasticNet epoch 20/20 완료
[0]	validation_0-rmse:0.22369
[100]	validation_0-rmse:0.22366
[200]	validation_0-rmse:0.22365
[300]	validation_0-rmse:0.22365
[400]	validation_0-rmse:0.22363
[500]	validation_0-rmse:0.22363
[600]	validation_0-rmse:0.22363
[603]	validation_0-rmse:0.22363

========== ridge Validation Result ==========
RMSE: 0.223646
MAE: 0.150844
per-era CORR mean: 0.024467
per-era CORR std:  0.011884
Sharpe:            2.058819

========== elasticnet Validation Result ==========
RMSE: 0.225329
MAE: 0.1

,model,rmse,mae,mean_corr,std_corr,sharpe
0,xgboost,0.223625,0.151929,0.023655,0.009544,2.478508
1,elasticnet,0.225329,0.163319,0.026658,0.011869,2.246006
2,ridge,0.223646,0.150844,0.024467,0.011884,2.058819


In [15]:
# ============================================
# 12. Ridge(alpha=1.0) 메타모델 + OOS 평가
# 세 기본 모델의 era별 rank를 메타 feature로 사용
# ============================================
meta_df = pd.DataFrame({
    "id": val_id_arr,
    "era": val_era_arr,
    "target": y_valid_arr,
    "ridge_prediction": ridge_pred,
    "elasticnet_prediction": elastic_pred,
    "xgboost_prediction": xgb_pred,
})

for model_name in ["ridge", "elasticnet", "xgboost"]:
    meta_df[f"{model_name}_rank"] = (
        meta_df.groupby("era")[f"{model_name}_prediction"]
        .rank(method="average", pct=True)
        .astype(np.float32)
    )

META_FEATURES = [
    "ridge_rank",
    "elasticnet_rank",
    "xgboost_rank",
]

# 앞 14 eras 학습 / 4 eras purge / 마지막 5 eras 평가
meta_eras = sorted(meta_df["era"].unique())
N_TEST_ERAS = 5
PURGE_ERAS = 4

meta_test_eras = meta_eras[-N_TEST_ERAS:]
purge_eras = meta_eras[-(N_TEST_ERAS + PURGE_ERAS):-N_TEST_ERAS]
meta_fit_eras = meta_eras[:-(N_TEST_ERAS + PURGE_ERAS)]

meta_fit_df = meta_df[meta_df["era"].isin(meta_fit_eras)].copy()
meta_test_df = meta_df[meta_df["era"].isin(meta_test_eras)].copy()

meta_model_oos = Ridge(alpha=1.0, fit_intercept=True)
meta_model_oos.fit(
    meta_fit_df[META_FEATURES],
    meta_fit_df["target"],
)

meta_test_df["stack_raw"] = meta_model_oos.predict(
    meta_test_df[META_FEATURES]
).astype(np.float32)

meta_test_df["stack_prediction"] = (
    meta_test_df.groupby("era")["stack_raw"]
    .rank(method="average", pct=True)
    .astype(np.float32)
)


def summarize_test(name, column):
    corr = simple_per_era_corr(
        meta_test_df["era"].to_numpy(),
        meta_test_df["target"].to_numpy(),
        meta_test_df[column].to_numpy(),
    )

    mean_corr = corr.mean()
    std_corr = corr.std(ddof=0)

    return {
        "model": name,
        "mean_corr": mean_corr,
        "std_corr": std_corr,
        "sharpe": mean_corr / std_corr if std_corr > 0 else np.nan,
    }


oos_results = pd.DataFrame([
    summarize_test("Ridge", "ridge_rank"),
    summarize_test("ElasticNet", "elasticnet_rank"),
    summarize_test("XGBoost", "xgboost_rank"),
    summarize_test(
        "Ridge + ElasticNet + XGBoost",
        "stack_prediction",
    ),
]).sort_values("sharpe", ascending=False).reset_index(drop=True)

print("메타 학습 era:", meta_fit_eras)
print("Purge era:", purge_eras)
print("메타 평가 era:", meta_test_eras)
print("OOS meta coefficients:", meta_model_oos.coef_)

display(oos_results)

meta_test_df[[
    "id",
    "era",
    "target",
    "ridge_rank",
    "elasticnet_rank",
    "xgboost_rank",
    "stack_prediction",
]].to_csv(
    MODEL_DIR / "ridge_elastic_xgboost_oos_predictions.csv",
    index=False,
)

joblib.dump(meta_model_oos, MODEL_DIR / "meta_ridge_alpha1_oos.joblib")

메타 학습 era: [np.int32(552), np.int32(553), np.int32(554), np.int32(555), np.int32(556), np.int32(557), np.int32(558), np.int32(559), np.int32(560), np.int32(561), np.int32(562), np.int32(563), np.int32(564), np.int32(565)]
Purge era: [np.int32(566), np.int32(567), np.int32(568), np.int32(569)]
메타 평가 era: [np.int32(570), np.int32(571), np.int32(572), np.int32(573), np.int32(574)]
OOS meta coefficients: [0.00135188 0.01433213 0.01274321]


,model,mean_corr,std_corr,sharpe
0,Ridge,0.016567,0.005003,3.311607
1,Ridge + ElasticNet + XGBoost,0.019217,0.006163,3.118258
2,ElasticNet,0.018825,0.006687,2.814968
3,XGBoost,0.014675,0.005389,2.723115


['/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/ridge_elastic_xgboost_stack_v52/meta_ridge_alpha1_oos.joblib']

In [16]:
# ============================================
# 13. 최종 메타모델 + 기본 모델 전체 내부 데이터 재학습
# ============================================
# 메타모델은 내부 validation 전체의 세 rank feature로 학습합니다.
final_meta_model = Ridge(alpha=1.0, fit_intercept=True)
final_meta_model.fit(
    meta_df[META_FEATURES],
    meta_df["target"],
)

print("Final meta coefficients:", final_meta_model.coef_)
print("Final meta intercept:", final_meta_model.intercept_)

# 내부 train/valid 원본 feature를 하나의 disk memmap으로 합칩니다.
X_FULL_RAW_PATH = MEMMAP_DIR / "X_full_raw.dat"
full_shape = (
    X_train.shape[0] + X_valid.shape[0],
    X_train.shape[1],
)

X_full = np.memmap(
    X_FULL_RAW_PATH,
    dtype="float32",
    mode="w+",
    shape=full_shape,
)
X_full[:X_train.shape[0]] = X_train
X_full[X_train.shape[0]:] = X_valid
X_full.flush()

y_full = np.concatenate([y_train_arr, y_valid_arr]).astype(np.float32)

# Ridge / ElasticNet 최종 입력 scaler
final_model_scaler = StandardScaler()

for start in range(0, X_full.shape[0], MODEL_BATCH_SIZE):
    end = min(start + MODEL_BATCH_SIZE, X_full.shape[0])
    final_model_scaler.partial_fit(X_full[start:end])
    print(f"Final model scaler: {end:,}/{X_full.shape[0]:,}")

X_FULL_SCALED_PATH = MEMMAP_DIR / "X_full_scaled.dat"
X_full_scaled = np.memmap(
    X_FULL_SCALED_PATH,
    dtype="float32",
    mode="w+",
    shape=X_full.shape,
)

for start in range(0, X_full.shape[0], MODEL_BATCH_SIZE):
    end = min(start + MODEL_BATCH_SIZE, X_full.shape[0])
    X_full_scaled[start:end] = final_model_scaler.transform(
        X_full[start:end]
    ).astype(np.float32)
    print(f"Final full scaling: {end:,}/{X_full.shape[0]:,}")

X_full_scaled.flush()

# Final Ridge
final_ridge_model = Ridge(
    alpha=BASE_RIDGE_ALPHA,
    fit_intercept=True,
    solver="lsqr",
    max_iter=5000,
    tol=1e-4,
)
final_ridge_model.fit(X_full_scaled, y_full)

# Final ElasticNet
final_elastic_model = SGDRegressor(
    loss="squared_error",
    penalty="elasticnet",
    alpha=1e-3,
    l1_ratio=0.03,
    learning_rate="invscaling",
    eta0=0.001,
    power_t=0.5,
    max_iter=1,
    tol=None,
    random_state=SEED,
    average=True,
)

rng = np.random.default_rng(SEED)

for epoch in range(ELASTIC_EPOCHS):
    indices = rng.permutation(X_full_scaled.shape[0])

    for start in range(0, len(indices), MODEL_BATCH_SIZE):
        end = min(start + MODEL_BATCH_SIZE, len(indices))
        idx = indices[start:end]

        final_elastic_model.partial_fit(
            np.asarray(X_full_scaled[idx], dtype=np.float32),
            y_full[idx],
        )

    print(f"Final ElasticNet epoch {epoch + 1}/{ELASTIC_EPOCHS} 완료")

# Final XGBoost
# 내부 early stopping에서 선택된 트리 수를 전체 재학습에 사용합니다.
if getattr(base_xgb_model, "best_iteration", None) is not None:
    FINAL_XGB_ESTIMATORS = int(base_xgb_model.best_iteration) + 1
else:
    FINAL_XGB_ESTIMATORS = int(XGB_BEST_PARAMS["n_estimators"])

final_xgb_params = XGB_BEST_PARAMS.copy()
final_xgb_params.pop("early_stopping_rounds", None)
final_xgb_params["n_estimators"] = FINAL_XGB_ESTIMATORS

final_xgb_model = xgb.XGBRegressor(**final_xgb_params)

try:
    final_xgb_model.fit(
        X_full,
        y_full,
        verbose=100,
    )
except Exception as error:
    if final_xgb_params.get("device") != "cuda":
        raise

    print("Final XGBoost CUDA 학습 실패 — CPU로 다시 실행합니다.")
    print("원래 오류:", error)

    final_xgb_params["device"] = "cpu"
    final_xgb_model = xgb.XGBRegressor(**final_xgb_params)
    final_xgb_model.fit(
        X_full,
        y_full,
        verbose=100,
    )

print("Final XGBoost tree 수:", FINAL_XGB_ESTIMATORS)

# 최종 객체 저장
joblib.dump(
    final_model_scaler,
    MODEL_DIR / "final_model_input_scaler.joblib",
)
joblib.dump(
    final_ridge_model,
    MODEL_DIR / "final_ridge_model.joblib",
)
joblib.dump(
    final_elastic_model,
    MODEL_DIR / "final_elasticnet_model.joblib",
)
joblib.dump(
    final_xgb_model,
    MODEL_DIR / "final_xgboost_model.joblib",
)
joblib.dump(
    final_meta_model,
    MODEL_DIR / "final_meta_ridge_model.joblib",
)

final_xgb_model.save_model(
    MODEL_DIR / "final_xgboost_model.json"
)

with open(MODEL_DIR / "final_features.json", "w", encoding="utf-8") as file:
    json.dump(FEATURES, file, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "xgboost_final_params.json", "w", encoding="utf-8") as file:
    json.dump(final_xgb_params, file, ensure_ascii=False, indent=2)

print("최종 모델 저장 완료:", MODEL_DIR)

Final meta coefficients: [-0.01328409  0.02733758  0.01115812]
Final meta intercept: 0.4873461
Final model scaler: 50,000/587,583
Final model scaler: 100,000/587,583
Final model scaler: 150,000/587,583
Final model scaler: 200,000/587,583
Final model scaler: 250,000/587,583
Final model scaler: 300,000/587,583
Final model scaler: 350,000/587,583
Final model scaler: 400,000/587,583
Final model scaler: 450,000/587,583
Final model scaler: 500,000/587,583
Final model scaler: 550,000/587,583
Final model scaler: 587,583/587,583
Final full scaling: 50,000/587,583
Final full scaling: 100,000/587,583
Final full scaling: 150,000/587,583
Final full scaling: 200,000/587,583
Final full scaling: 250,000/587,583
Final full scaling: 300,000/587,583
Final full scaling: 350,000/587,583
Final full scaling: 400,000/587,583
Final full scaling: 450,000/587,583
Final full scaling: 500,000/587,583
Final full scaling: 550,000/587,583
Final full scaling: 587,583/587,583
Final ElasticNet epoch 1/20 완료
Final Elasti

In [17]:
# ============================================
# 14. 공식 validation.parquet → PCA500
# ============================================
if not VALIDATION_PCA_ALL_PATH.exists():
    transform_source_to_pca(
        VALIDATION_PATH,
        VALIDATION_PCA_ALL_PATH,
        filter_validation=True,
    )
else:
    print("기존 파일 사용:", VALIDATION_PCA_ALL_PATH)

기존 파일 사용: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/cache_v52/validation_pca_all.parquet


In [18]:
# ============================================
# 15. 공식 validation PCA500 → lag1/lag2 → 1500 features
# ============================================
validation_era_mean_df = compute_era_means(VALIDATION_PCA_ALL_PATH)
validation_eras = sorted(
    validation_era_mean_df.index.astype(int).tolist()
)
validation_era_set = set(validation_eras)

validation_lag1, validation_lag2 = make_lag_lookup(
    validation_era_mean_df,
    validation_era_set,
)

remove_if_exists(VALIDATION_FINAL_PATH)
validation_writer = {
    "path": VALIDATION_FINAL_PATH,
    "object": None,
}
validation_rows = 0

try:
    for chunk in iter_parquet_batches(
        VALIDATION_PCA_ALL_PATH,
        ["id", "era", TARGET_COL] + pca_cols,
        TRANSFORM_BATCH_SIZE,
    ):
        era_array = chunk["era"].to_numpy(dtype=np.int32)

        lag1_array = np.vstack([
            validation_lag1[int(era)]
            for era in era_array
        ])
        lag2_array = np.vstack([
            validation_lag2[int(era)]
            for era in era_array
        ])

        final = pd.concat(
            [
                chunk[["id"] + pca_cols].reset_index(drop=True),
                pd.DataFrame(lag1_array, columns=lag1_cols),
                pd.DataFrame(lag2_array, columns=lag2_cols),
            ],
            axis=1,
        )

        final["era"] = era_array
        final[TARGET_COL] = chunk[TARGET_COL].to_numpy(dtype=np.float32)
        final = final[["id"] + FEATURES + ["era", TARGET_COL]]

        append_parquet_batch(validation_writer, final)
        validation_rows += len(final)
        print(f"validation final: {validation_rows:,} rows")

finally:
    close_writer(validation_writer)

print("공식 validation 전처리 저장:", VALIDATION_FINAL_PATH)
print("era:", min(validation_eras), "~", max(validation_eras))

validation final: 20,000 rows
validation final: 40,000 rows
validation final: 60,000 rows
validation final: 80,000 rows
validation final: 100,000 rows
validation final: 120,000 rows
validation final: 140,000 rows
validation final: 160,000 rows
validation final: 180,000 rows
validation final: 200,000 rows
validation final: 220,000 rows
validation final: 240,000 rows
validation final: 260,000 rows
validation final: 280,000 rows
validation final: 300,000 rows
validation final: 320,000 rows
validation final: 340,000 rows
validation final: 360,000 rows
validation final: 380,000 rows
validation final: 400,000 rows
validation final: 420,000 rows
validation final: 440,000 rows
validation final: 460,000 rows
validation final: 480,000 rows
validation final: 500,000 rows
validation final: 520,000 rows
validation final: 540,000 rows
validation final: 560,000 rows
validation final: 580,000 rows
validation final: 600,000 rows
validation final: 620,000 rows
validation final: 640,000 rows
validation f

In [19]:
# ============================================
# 16. 공식 validation 예측 + Diagnostics CSV + 로컬 CORR
# era가 끝날 때마다 저장하여 메모리를 절약합니다.
# ============================================
DIAGNOSTICS_CSV = (
    DIAGNOSTICS_DIR
    / "ridge_elastic_xgboost_v52_diagnostics.csv"
)
PER_ERA_CSV = (
    DIAGNOSTICS_DIR
    / "ridge_elastic_xgboost_v52_per_era_corr.csv"
)
SUMMARY_CSV = (
    DIAGNOSTICS_DIR
    / "ridge_elastic_xgboost_v52_local_summary.csv"
)

remove_if_exists(DIAGNOSTICS_CSV)

state = {
    "era": None,
    "id": [],
    "target": [],
    "ridge": [],
    "elastic": [],
    "xgboost": [],
}

per_era_rows = []
header_written = False
last_seen_era = None


def flush_era_state():
    global header_written

    if state["era"] is None:
        return

    era_df = pd.DataFrame({
        "id": np.concatenate(state["id"]),
        "target": np.concatenate(state["target"]),
        "ridge_prediction": np.concatenate(state["ridge"]),
        "elasticnet_prediction": np.concatenate(state["elastic"]),
        "xgboost_prediction": np.concatenate(state["xgboost"]),
    })

    for model_name in ["ridge", "elasticnet", "xgboost"]:
        source_column = (
            "elasticnet_prediction"
            if model_name == "elasticnet"
            else f"{model_name}_prediction"
        )
        era_df[f"{model_name}_rank"] = (
            era_df[source_column]
            .rank(method="average", pct=True)
            .astype(np.float32)
        )

    meta_X = era_df[META_FEATURES].to_numpy(dtype=np.float32)
    era_df["stack_raw"] = final_meta_model.predict(meta_X).astype(np.float32)
    era_df["prediction"] = (
        era_df["stack_raw"]
        .rank(method="average", pct=True)
        .astype(np.float32)
    )

    score_data = era_df[["prediction", "target"]].dropna()
    score = numerai_corr(
        score_data[["prediction"]],
        score_data["target"],
    )
    score_value = float(np.asarray(score).reshape(-1)[0])

    era_df[["id", "prediction"]].to_csv(
        DIAGNOSTICS_CSV,
        mode="a",
        header=not header_written,
        index=False,
    )
    header_written = True

    per_era_rows.append({
        "era": int(state["era"]),
        "corr": score_value,
        "rows": len(era_df),
    })

    print(
        f"era {state['era']} 저장 | "
        f"rows={len(era_df):,} | CORR={score_value:.6f}"
    )

    state["era"] = None
    state["id"] = []
    state["target"] = []
    state["ridge"] = []
    state["elastic"] = []
    state["xgboost"] = []


for chunk in iter_parquet_batches(
    VALIDATION_FINAL_PATH,
    ["id", "era", TARGET_COL] + FEATURES,
    TRANSFORM_BATCH_SIZE,
):
    chunk = chunk.reset_index(drop=True)

    X_model = clean_array(
        chunk[FEATURES].to_numpy(dtype=np.float32)
    )
    X_scaled = final_model_scaler.transform(X_model).astype(np.float32)

    ridge_chunk = final_ridge_model.predict(X_scaled).astype(np.float32)
    elastic_chunk = final_elastic_model.predict(X_scaled).astype(np.float32)
    xgboost_chunk = final_xgb_model.predict(X_model).astype(np.float32)

    for era, positions in chunk.groupby("era", sort=False).groups.items():
        era = int(era)
        positions = np.asarray(list(positions), dtype=int)

        if last_seen_era is not None and era < last_seen_era:
            raise ValueError(
                "validation parquet이 era 순서가 아닙니다. "
                "먼저 era 기준 정렬 파일을 만들어야 합니다."
            )

        last_seen_era = era

        if state["era"] is not None and era != state["era"]:
            flush_era_state()

        if state["era"] is None:
            state["era"] = era

        state["id"].append(
            chunk.iloc[positions]["id"].astype(str).to_numpy()
        )
        state["target"].append(
            chunk.iloc[positions][TARGET_COL].to_numpy(dtype=np.float32)
        )
        state["ridge"].append(ridge_chunk[positions])
        state["elastic"].append(elastic_chunk[positions])
        state["xgboost"].append(xgboost_chunk[positions])

flush_era_state()

per_era_df = pd.DataFrame(per_era_rows).sort_values("era")
per_era_df.to_csv(PER_ERA_CSV, index=False)

corr_mean = per_era_df["corr"].mean()
corr_std = per_era_df["corr"].std(ddof=0)
corr_sharpe = corr_mean / corr_std if corr_std > 0 else np.nan

cumulative = per_era_df["corr"].cumsum()
max_drawdown = (
    cumulative.expanding(min_periods=1).max() - cumulative
).max()

summary_df = pd.DataFrame({
    "metric": [
        "mean_corr",
        "std_corr",
        "sharpe",
        "max_drawdown",
        "eras",
    ],
    "value": [
        corr_mean,
        corr_std,
        corr_sharpe,
        max_drawdown,
        len(per_era_df),
    ],
})
summary_df.to_csv(SUMMARY_CSV, index=False)

display(summary_df)
print("Diagnostics CSV:", DIAGNOSTICS_CSV)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


era 575 저장 | rows=5,592 | CORR=0.020510
era 576 저장 | rows=5,638 | CORR=0.010800
era 577 저장 | rows=5,675 | CORR=0.009182
era 578 저장 | rows=5,786 | CORR=0.014243


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: Us

era 579 저장 | rows=5,868 | CORR=0.016020
era 580 저장 | rows=5,875 | CORR=0.013124
era 581 저장 | rows=5,840 | CORR=0.010940
era 582 저장 | rows=5,819 | CORR=0.017319
era 583 저장 | rows=5,814 | CORR=0.030153


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


...

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


era 1222 저장 | rows=7,105 | CORR=0.010082
era 1223 저장 | rows=7,102 | CORR=0.010783


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


,metric,value
0,mean_corr,0.010166
1,std_corr,0.013488
2,sharpe,0.753735
3,max_drawdown,0.188256
4,eras,649.000000


Diagnostics CSV: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/ridge_elastic_xgboost_v52_diagnostics.csv


In [20]:
# ============================================
# 17. 업로드 파일 최종 검사
# ============================================
diagnostics_check = pd.read_csv(DIAGNOSTICS_CSV)

print("shape:", diagnostics_check.shape)
print(diagnostics_check.head())
print(diagnostics_check["prediction"].describe())

assert diagnostics_check.columns.tolist() == ["id", "prediction"]
assert diagnostics_check["id"].is_unique
assert diagnostics_check["prediction"].notna().all()
assert diagnostics_check["prediction"].between(0, 1).all()

print("Diagnostics 업로드 형식 확인 완료")

shape: (4050241, 2)
                 id  prediction
0  n000101811a8a843    0.151645
1  n001e1318d5072ac    0.867668
2  n002a9c5ab785cbb    0.095136
3  n002ccf6d0e8c5ad    0.427754
4  n0041544c345c91d    0.099070
count    4.050241e+06
mean     5.000801e-01
std      2.886752e-01
min      1.368176e-04
25%      2.500804e-01
50%      5.000812e-01
75%      7.500795e-01
max      1.000000e+00
Name: prediction, dtype: float64
Diagnostics 업로드 형식 확인 완료


## 생성되는 주요 파일

### 전처리 객체
- `numerai_postprocess_v52/selected_features.json`
- `numerai_postprocess_v52/fitted_scaler.joblib`
- `numerai_postprocess_v52/fitted_pca.joblib`

### 캐시
- `cache_v52/train_pca_all.parquet`
- `cache_v52/train_final_cache.parquet`
- `cache_v52/val_final_cache.parquet`
- `pca500_lag1_lag2.parquet`
- `cache_v52/validation_pca_all.parquet`
- `cache_v52/validation_pca500_lag1_lag2.parquet`

### 모델 / 내부 예측
- `ridge_elastic_xgboost_stack_v52/base_ridge_model.joblib`
- `ridge_elastic_xgboost_stack_v52/base_elasticnet_model.joblib`
- `ridge_elastic_xgboost_stack_v52/base_xgboost_model.json`
- `ridge_elastic_xgboost_stack_v52/final_ridge_model.joblib`
- `ridge_elastic_xgboost_stack_v52/final_elasticnet_model.joblib`
- `ridge_elastic_xgboost_stack_v52/final_xgboost_model.json`
- `ridge_elastic_xgboost_stack_v52/final_meta_ridge_model.joblib`
- 각 모델의 `*_realtarget_val_predictions.csv`

### Numerai Diagnostics
- `diagnostics_v52/ridge_elastic_xgboost_v52_diagnostics.csv`
- `diagnostics_v52/ridge_elastic_xgboost_v52_per_era_corr.csv`
- `diagnostics_v52/ridge_elastic_xgboost_v52_local_summary.csv`

Scores 페이지의 플라스크 아이콘에는 `ridge_elastic_xgboost_v52_diagnostics.csv`를 업로드합니다.